In [ ]:
%%writefile train_model.py
"""Train an EfficientNetB0 classifier for Project 2 (16 Pakistani public figures).

- Uses transfer learning (ImageNet weights) with frozen EfficientNetB0 backbone.
- Builds a small custom head: GAP -> Dropout -> Dense(softmax).
- Creates stratified per-class train/val/test splits from dataset directory.
- Applies required data augmentation on training data only.
- Logs params/metrics/artifacts to MLflow.

Expected dataset layout (preferred):
  dataset/train/<class_name>/*
  dataset/val/<class_name>/*
  dataset/test/<class_name>/*

If split folders do not exist, the script will fall back to:
  dataset/raw/<class_name>/*

and will create an in-memory 75/15/10 split (stratified per class).
"""

from __future__ import annotations

import argparse
import json
import os
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable, List, Sequence, Tuple


def _np():
    import numpy as np

    return np


def _strip_jupyter_kernel_args(argv: Sequence[str]) -> List[str]:
    """Remove ipykernel/colab connection-file args from an argv list.

    In notebooks, the running kernel process often has arguments like:
      -f /path/to/kernel-<uuid>.json

    If this training script is imported/executed in that environment and we
    reuse sys.argv, argparse will otherwise treat that JSON path as an
    unrecognized argument.
    """

    cleaned: List[str] = []
    skip_next = False
    for arg in argv:
        if skip_next:
            skip_next = False
            continue

        if arg in {"-f", "--f"}:
            skip_next = True
            continue

        if arg.startswith("-f=") or arg.startswith("--f="):
            continue

        # Defensive fallback: sometimes the connection file path is present
        # without the preceding flag.
        if arg.endswith(".json"):
            name = Path(arg).name
            if name.startswith("kernel-") and "jupyter" in arg and "runtime" in arg:
                continue

        cleaned.append(arg)

    return cleaned


def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train EfficientNetB0 on politician image dataset")
    parser.add_argument(
        "--dataset_root",
        type=Path,
        default=Path("dataset"),
        help="Dataset root. Uses dataset/{train,val,test} if present, else dataset/raw.",
    )
    parser.add_argument(
        "--output_dir",
        type=Path,
        default=Path("artifacts") / "efficientnetb0",
        help="Directory to write artifacts (model, plots).",
    )
    parser.add_argument("--img_size", type=int, default=224, help="Input image size (square).")
    parser.add_argument("--batch_size", type=int, default=16, help="Batch size.")
    parser.add_argument("--epochs", type=int, default=30, help="Max epochs.")
    parser.add_argument("--seed", type=int, default=42, help="Random seed.")
    parser.add_argument(
        "--learning_rate",
        type=float,
        default=1e-3,
        help="Learning rate for Adam optimizer.",
    )
    parser.add_argument(
        "--dropout",
        type=float,
        default=0.5,
        help="Dropout rate for the classification head.",
    )
    parser.add_argument(
        "--l2_reg",
        type=float,
        default=0.01,
        help="L2 regularization strength for Dense layer.",
    )
    parser.add_argument(
        "--mixup_alpha",
        type=float,
        default=0.2,
        help="Mixup augmentation alpha parameter. Set to 0 to disable mixup.",
    )
    parser.add_argument(
        "--experiment_name",
        type=str,
        default="project2-politician-classification",
        help="MLflow experiment name.",
    )
    parser.add_argument(
        "--run_name",
        type=str,
        default=None,
        help="Optional MLflow run name.",
    )
    parser.add_argument(
        "--mlflow_tracking_uri",
        type=str,
        default=os.environ.get("MLFLOW_TRACKING_URI", ""),
        help="MLflow tracking URI. Defaults to env MLFLOW_TRACKING_URI or local file store.",
    )
    parser.add_argument(
        "--no_mlflow",
        action="store_true",
        help="Disable MLflow logging (useful if mlflow isn't installed).",
    )
    if argv is None:
        argv = sys.argv[1:]
    argv = _strip_jupyter_kernel_args(list(argv))
    return parser.parse_args(argv)


IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


@dataclass(frozen=True)
class SplitData:
    train_paths: List[Path]
    train_labels: List[int]
    val_paths: List[Path]
    val_labels: List[int]
    test_paths: List[Path]
    test_labels: List[int]
    class_names: List[str]


def _list_class_dirs(root: Path) -> List[Path]:
    return sorted([p for p in root.iterdir() if p.is_dir()])


def _gather_images_per_class(class_dir: Path) -> List[Path]:
    images: List[Path] = []
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            images.append(p)
    return sorted(images)


def _stratified_split(
    class_dirs: Sequence[Path],
    seed: int,
    train_frac: float = 0.75,
    val_frac: float = 0.15,
    test_frac: float = 0.10,
) -> SplitData:
    np = _np()
    if not np.isclose(train_frac + val_frac + test_frac, 1.0):
        raise ValueError("train/val/test fractions must sum to 1.0")

    rng = np.random.default_rng(seed)

    class_names = [d.name for d in class_dirs]
    train_paths: List[Path] = []
    train_labels: List[int] = []
    val_paths: List[Path] = []
    val_labels: List[int] = []
    test_paths: List[Path] = []
    test_labels: List[int] = []

    for label, class_dir in enumerate(class_dirs):
        images = _gather_images_per_class(class_dir)
        if not images:
            raise ValueError(f"No images found in {class_dir}")

        idx = np.arange(len(images))
        rng.shuffle(idx)

        n_train = int(round(len(images) * train_frac))
        n_val = int(round(len(images) * val_frac))
        n_test = len(images) - n_train - n_val

        if n_train <= 0 or n_val <= 0 or n_test <= 0:
            raise ValueError(
                f"Not enough images for split in {class_dir} "
                f"(train={n_train}, val={n_val}, test={n_test})."
            )

        train_idx = idx[:n_train]
        val_idx = idx[n_train : n_train + n_val]
        test_idx = idx[n_train + n_val :]

        for i in train_idx:
            train_paths.append(images[int(i)])
            train_labels.append(label)
        for i in val_idx:
            val_paths.append(images[int(i)])
            val_labels.append(label)
        for i in test_idx:
            test_paths.append(images[int(i)])
            test_labels.append(label)

    return SplitData(
        train_paths=train_paths,
        train_labels=train_labels,
        val_paths=val_paths,
        val_labels=val_labels,
        test_paths=test_paths,
        test_labels=test_labels,
        class_names=class_names,
    )


def _load_split_from_directories(dataset_root: Path, seed: int) -> SplitData:
    np = _np()
    split_root_candidates = [
        (dataset_root / "train", dataset_root / "val", dataset_root / "test"),
        (dataset_root / "train", dataset_root / "valid", dataset_root / "test"),
        (dataset_root / "training", dataset_root / "validation", dataset_root / "test"),
    ]

    for train_dir, val_dir, test_dir in split_root_candidates:
        if train_dir.exists() and val_dir.exists() and test_dir.exists():
            train_class_dirs = _list_class_dirs(train_dir)
            val_class_dirs = _list_class_dirs(val_dir)
            test_class_dirs = _list_class_dirs(test_dir)

            train_class_names = [p.name for p in train_class_dirs]
            val_class_names = [p.name for p in val_class_dirs]
            test_class_names = [p.name for p in test_class_dirs]

            if train_class_names != val_class_names or train_class_names != test_class_names:
                raise ValueError(
                    "Class folders must match across train/val/test. "
                    f"Got train={train_class_names}, val={val_class_names}, test={test_class_names}."
                )

            class_names = train_class_names
            class_to_idx = {name: i for i, name in enumerate(class_names)}

            def read_split(split_dir: Path) -> Tuple[List[Path], List[int]]:
                paths: List[Path] = []
                labels: List[int] = []
                for class_dir in _list_class_dirs(split_dir):
                    label = class_to_idx[class_dir.name]
                    images = _gather_images_per_class(class_dir)
                    paths.extend(images)
                    labels.extend([label] * len(images))
                perm = np.random.default_rng(seed).permutation(len(paths))
                paths = [paths[i] for i in perm]
                labels = [labels[i] for i in perm]
                return paths, labels

            train_paths, train_labels = read_split(train_dir)
            val_paths, val_labels = read_split(val_dir)
            test_paths, test_labels = read_split(test_dir)

            return SplitData(
                train_paths=train_paths,
                train_labels=train_labels,
                val_paths=val_paths,
                val_labels=val_labels,
                test_paths=test_paths,
                test_labels=test_labels,
                class_names=class_names,
            )

    raw_dir = dataset_root / "raw"
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Could not find a dataset split under {dataset_root} and {raw_dir} does not exist."
        )

    class_dirs = _list_class_dirs(raw_dir)
    if not class_dirs:
        raise ValueError(f"No class folders found under {raw_dir}")

    return _stratified_split(class_dirs=class_dirs, seed=seed)


def _mixup(image: "tf.Tensor", label: "tf.Tensor", alpha: float = 0.2) -> Tuple["tf.Tensor", "tf.Tensor"]:
    """Apply mixup augmentation to a batch of images and labels."""
    import tensorflow as tf
    
    batch_size = tf.shape(image)[0]
    
    # Sample lambda from Beta distribution (simplified as uniform for efficiency)
    lam = tf.random.uniform([], 0, alpha)
    
    # Shuffle indices
    indices = tf.random.shuffle(tf.range(batch_size))
    
    # Mix images and labels
    mixed_image = lam * image + (1 - lam) * tf.gather(image, indices)
    mixed_label = lam * label + (1 - lam) * tf.gather(label, indices)
    
    return mixed_image, mixed_label


def _build_tf_dataset(
    paths: Sequence[Path],
    labels: Sequence[int],
    *,
    class_count: int,
    img_size: int,
    batch_size: int,
    seed: int,
    training: bool,
    mixup_alpha: float = 0.0,
):
    import tensorflow as tf

    ds_paths = tf.convert_to_tensor([str(p) for p in paths], dtype=tf.string)
    ds_labels = tf.convert_to_tensor(labels, dtype=tf.int32)

    ds = tf.data.Dataset.from_tensor_slices((ds_paths, ds_labels))

    if training:
        ds = ds.shuffle(buffer_size=min(len(paths), 2048), seed=seed, reshuffle_each_iteration=True)

    def load_and_preprocess(path: tf.Tensor, label: tf.Tensor):
        image_bytes = tf.io.read_file(path)
        image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.convert_image_dtype(image, dtype=tf.float32)  # [0, 1]

        if training:
            image = tf.image.resize(image, (img_size + 32, img_size + 32), method="bilinear")
            image = tf.image.random_crop(image, size=(img_size, img_size, 3))
        else:
            image = tf.image.resize(image, (img_size, img_size), method="bilinear")

        return image, label

    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    data_augmentation = None
    if training:
        # UPDATED: Stronger augmentation to combat overfitting
        data_augmentation = tf.keras.Sequential(
            [
                tf.keras.layers.RandomFlip("horizontal"),
                tf.keras.layers.RandomRotation(0.15),  # Increased from 0.10
                tf.keras.layers.RandomZoom(0.25),      # Increased from 0.20
                tf.keras.layers.RandomContrast(0.2),   # NEW: Added contrast augmentation
                tf.keras.layers.RandomTranslation(0.1, 0.1),  # NEW: Added translation
            ],
            name="data_augmentation",
        )

        def augment(image: tf.Tensor, label: tf.Tensor):
            image4 = tf.expand_dims(image, axis=0)
            image4 = data_augmentation(image4, training=True)
            image = tf.squeeze(image4, axis=0)
            image = tf.image.random_brightness(image, max_delta=0.20)
            image = tf.clip_by_value(image, 0.0, 1.0)
            return image, label

        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    def to_model_inputs(image: tf.Tensor, label: tf.Tensor):
        # EfficientNet preprocess_input expects [0, 255]
        image_255 = image * 255.0
        image_pp = tf.keras.applications.efficientnet.preprocess_input(image_255)
        label_oh = tf.one_hot(label, depth=class_count)
        return image_pp, label_oh

    ds = ds.map(to_model_inputs, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size)
    
    # UPDATED: Apply mixup augmentation after batching (if enabled for training)
    if training and mixup_alpha > 0:
        ds = ds.map(
            lambda x, y: _mixup(x, y, alpha=mixup_alpha),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def _plot_history(history, out_path: Path):
    import matplotlib.pyplot as plt

    hist = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    if "loss" in hist:
        ax[0].plot(hist["loss"], label="train")
    if "val_loss" in hist:
        ax[0].plot(hist["val_loss"], label="val")
    ax[0].set_title("Loss")
    ax[0].legend()

    if "accuracy" in hist:
        ax[1].plot(hist["accuracy"], label="train")
    if "val_accuracy" in hist:
        ax[1].plot(hist["val_accuracy"], label="val")
    ax[1].set_title("Accuracy")
    ax[1].legend()

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def _plot_confusion_matrix(cm: "np.ndarray", class_names: Sequence[str], out_path: Path):
    import matplotlib.pyplot as plt
    np = _np()

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title="Confusion Matrix",
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center", color="white" if cm[i, j] > thresh else "black")

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def _plot_misclassified(
    paths: Sequence[Path],
    y_true: Sequence[int],
    y_pred: Sequence[int],
    class_names: Sequence[str],
    out_path: Path,
    max_samples: int = 5,
):
    import matplotlib.pyplot as plt

    wrong = [(p, t, pr) for p, t, pr in zip(paths, y_true, y_pred) if int(t) != int(pr)]
    wrong = wrong[:max_samples]

    if not wrong:
        return False

    fig, axes = plt.subplots(1, len(wrong), figsize=(4 * len(wrong), 4))
    if len(wrong) == 1:
        axes = [axes]

    for ax, (p, t, pr) in zip(axes, wrong):
        import tensorflow as tf

        img = tf.io.read_file(str(p))
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        ax.imshow(img.numpy())
        ax.axis("off")
        ax.set_title(f"T: {class_names[int(t)]}\nP: {class_names[int(pr)]}")

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160)
    plt.close(fig)
    return True


def main() -> int:
    args = parse_args()

    import tensorflow as tf
    np = _np()
    from sklearn.metrics import classification_report, confusion_matrix

    class MlflowKerasCallback(tf.keras.callbacks.Callback):
        def __init__(self):
            super().__init__()
            import mlflow

            self._mlflow = mlflow

        def on_epoch_end(self, epoch, logs=None):
            if not logs:
                return
            metrics = {k: float(v) for k, v in logs.items() if v is not None}
            self._mlflow.log_metrics(metrics, step=int(epoch))

    tf.keras.utils.set_random_seed(args.seed)

    split = _load_split_from_directories(args.dataset_root, seed=args.seed)
    class_count = len(split.class_names)

    train_ds = _build_tf_dataset(
        split.train_paths,
        split.train_labels,
        class_count=class_count,
        img_size=args.img_size,
        batch_size=args.batch_size,
        seed=args.seed,
        training=True,
        mixup_alpha=args.mixup_alpha,
    )
    val_ds = _build_tf_dataset(
        split.val_paths,
        split.val_labels,
        class_count=class_count,
        img_size=args.img_size,
        batch_size=args.batch_size,
        seed=args.seed,
        training=False,
        mixup_alpha=0.0,
    )
    test_ds = _build_tf_dataset(
        split.test_paths,
        split.test_labels,
        class_count=class_count,
        img_size=args.img_size,
        batch_size=args.batch_size,
        seed=args.seed,
        training=False,
        mixup_alpha=0.0,
    )

    # 1. Load the EfficientNetB0 base
    base_model = tf.keras.applications.EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(args.img_size, args.img_size, 3),
    )

    # 2. Unfreeze the base model for Fine-Tuning
    base_model.trainable = True

    # 3. Re-freeze the bottom layers (leave top 40 unfrozen)
    for layer in base_model.layers[:-40]:
        layer.trainable = False

    # 4. Build the custom classification head (NO L2 Regularization here!)
    x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
    x = tf.keras.layers.Dropout(args.dropout)(x)
    outputs = tf.keras.layers.Dense(class_count, activation="softmax")(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=args.learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    run_dir = args.output_dir / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_path = run_dir / "model_summary.txt"
    with summary_path.open("w", encoding="utf-8") as f:
        model.summary(print_fn=lambda line: f.write(line + "\n"))

    class_map_path = run_dir / "class_names.json"
    class_map_path.write_text(json.dumps(split.class_names, indent=2), encoding="utf-8")

    best_model_path = run_dir / "best_model.keras"

    callbacks: List[tf.keras.callbacks.Callback] = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=6,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-6,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_path),
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]

    mlflow_enabled = (not args.no_mlflow)
    mlflow = None
    if mlflow_enabled:
        try:
            import mlflow  # type: ignore

            if args.mlflow_tracking_uri:
                mlflow.set_tracking_uri(args.mlflow_tracking_uri)
            mlflow.set_experiment(args.experiment_name)
        except Exception:
            mlflow_enabled = False

    if mlflow_enabled:
        callbacks.append(MlflowKerasCallback())

    params_to_log = {
        "model": "EfficientNetB0",
        "img_size": args.img_size,
        "batch_size": args.batch_size,
        "epochs": args.epochs,
        "learning_rate": args.learning_rate,
        "dropout": args.dropout,
        "l2_reg": args.l2_reg,
        "mixup_alpha": args.mixup_alpha,
        "seed": args.seed,
        "classes": class_count,
        "train_samples": len(split.train_paths),
        "val_samples": len(split.val_paths),
        "test_samples": len(split.test_paths),
        "dataset_root": str(args.dataset_root),
    }

    def train_and_evaluate():
        history = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs, callbacks=callbacks)

        curves_path = run_dir / "training_curves.png"
        _plot_history(history, curves_path)

        test_loss, test_acc = model.evaluate(test_ds, verbose=0)

        y_true = np.array(split.test_labels, dtype=np.int64)
        y_probs = model.predict(test_ds, verbose=0)
        y_pred = np.argmax(y_probs, axis=1)

        cm = confusion_matrix(y_true, y_pred, labels=list(range(class_count)))
        cm_path = run_dir / "confusion_matrix.png"
        _plot_confusion_matrix(cm, split.class_names, cm_path)

        report = classification_report(
            y_true,
            y_pred,
            labels=list(range(class_count)),
            target_names=split.class_names,
            output_dict=True,
            zero_division=0,
        )
        report_path = run_dir / "classification_report.json"
        report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

        misclassified_path = run_dir / "misclassified_top5.png"
        misclassified_written = _plot_misclassified(
            split.test_paths,
            y_true,
            y_pred,
            split.class_names,
            misclassified_path,
            max_samples=5,
        )

        model.save(run_dir / "final_model.keras")

        return {
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "curves_path": curves_path,
            "cm_path": cm_path,
            "report_path": report_path,
            "class_map_path": class_map_path,
            "best_model_path": best_model_path,
            "final_model_path": run_dir / "final_model.keras",
            "misclassified_path": misclassified_path if misclassified_written else None,
        }

    if mlflow_enabled and mlflow is not None:
        with mlflow.start_run(run_name=args.run_name):
            mlflow.log_params(params_to_log)
            results = train_and_evaluate()
            mlflow.log_metrics(
                {
                    "test_loss": results["test_loss"],
                    "test_accuracy": results["test_accuracy"],
                }
            )
            mlflow.log_artifact(str(results["curves_path"]))
            mlflow.log_artifact(str(results["cm_path"]))
            mlflow.log_artifact(str(results["report_path"]))
            mlflow.log_artifact(str(summary_path))
            mlflow.log_artifact(str(results["class_map_path"]))
            mlflow.log_artifact(str(results["best_model_path"]))
            mlflow.log_artifact(str(results["final_model_path"]))
            if results["misclassified_path"] is not None:
                mlflow.log_artifact(str(results["misclassified_path"]))
    else:
        train_and_evaluate()

    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
!python train_model.py \
    --dataset_root /kaggle/working/clean_dataset \
    --output_dir /kaggle/working/artifacts \
    --epochs 40 \
    --batch_size 16 \
    --learning_rate 1e-4 \
    --dropout 0.3 \
    --mixup_alpha 0.0 \
    --no_mlflow